# 🎧 Otimização Sistemática de Prompts — do rascunho à produção

### Um caso real de central de atendimento

> **A história da Aurora** 🌅
>
> A **TechZé** é uma loja online de eletrônicos. Todo dia chegam **milhares de
> mensagens** de clientes: elogios, reclamações, dúvidas, "cadê meu pedido???".
>
> A equipe de suporte é pequena e vive apagando incêndio. Ninguém consegue ler
> tudo a tempo, e as reclamações mais urgentes às vezes ficam esperando horas
> no meio de um monte de "ok, obrigado".
>
> A **Aurora** é a assistente de IA que o time acabou de contratar. A primeira
> missão dela é simples de dizer e traiçoeira de fazer: **ler cada mensagem e
> dizer se o sentimento é `positivo`, `negativo` ou `neutro`**, para que as
> mensagens negativas furem a fila e cheguem primeiro a um humano.
>
> Parece fácil. Mas a Aurora só é tão boa quanto o **prompt** que a guia. E aqui
> começa a nossa jornada: transformar um prompt "que mais ou menos funciona" em
> um prompt **medido, comparado e pronto para produção**.

Neste notebook a gente vai seguir, passo a passo, os slides da aula e viver o
ciclo completo com a Aurora:

1. **Versionamento de prompts + Teste A/B** — comparar prompts como quem compara
   receitas.
2. **Otimização sistemática** — métrica → dataset → baseline → iteração → decisão.
3. **Checklist de produção responsável** — o que precisa estar de pé *antes* de
   soltar a Aurora no mundo (guardrails, LGPD, observabilidade, rollback).

> 💡 Tudo aqui roda **sem chave de API**: usamos um *simulador de LLM* para a
> aula ser reprodutível. No final, uma única chave (`BACKEND`) troca o simulador
> por um modelo real — **Google Gemini**, Anthropic ou Azure OpenAI.

## 🔧 Preparando o terreno: a "LLM" da Aurora

Antes de tudo, precisamos de algo que responda como um modelo responderia.

Em produção, `chamar_llm(prompt)` faria uma chamada real à API. Para a aula,
usamos um **simulador**: ele lê o prompt, percebe o quão bem escrito ele está
(se define a classe `neutro`, se traz exemplos, se pede formato de saída) e
responde de forma mais ou menos acertada — **exatamente como um modelo real, que
melhora quando o prompt melhora.**

O contrato da função é o mesmo dos slides: entra um `prompt` (string), sai um
`rótulo` (string).

In [1]:
import re
import time
import random
import pandas as pd

random.seed(42)

CLASSES_VALIDAS = {"positivo", "negativo", "neutro"}

# ==================== CONFIGURAÇÃO DO BACKEND ====================
# "simulador" -> roda sem chave de API (padrão didático, reprodutível)
# "gemini"    -> Google AI / Gemini   (pip install google-genai)
# "anthropic" -> Claude               (pip install anthropic)
# "azure"     -> Azure OpenAI         (pip install openai)
import os
BACKEND        = "simulador"
GEMINI_MODEL   = "gemini-2.5-flash"   # rápido/barato p/ classificar; ou "gemini-3.5-flash"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")   # ou cole a chave entre as aspas
# ================================================================

# --- Preço e velocidade fictícios só para ilustrar o trade-off de custo/latência
PRECO_POR_1K_TOKENS = 0.03     # R$ por 1.000 tokens de entrada (valor didático)
MS_POR_TOKEN        = 1.4      # latência simulada por token
LATENCIA_BASE_MS    = 40


def _estimar_tokens(texto: str) -> int:
    """Aproximação grosseira: ~4 caracteres por token."""
    return max(1, len(texto) // 4)


def _detectar_capacidades(prompt: str) -> dict:
    """Lê o prompt e infere quais 'boas práticas' ele usa.

    É isso que faz o simulador ser realista: um prompt melhor -> respostas
    melhores, do mesmo jeito que acontece com um modelo de verdade.
    """
    p = prompt.lower()
    return {
        "neutro":   "neutro" in p,                                   # define a 3ª classe?
        "exemplos": ("exemplo" in p) or ("->" in p),                 # tem few-shot?
        "schema":   ("apenas" in p and "palavra" in p) or ("sem explica" in p),
    }


def _heuristica_sentimento(texto: str) -> str:
    """O 'raciocínio' base do modelo simulado."""
    t = texto.lower()
    positivos = ["adorei", "amei", "melhor", "ótimo", "otimo", "excelente",
                 "perfeito", "gostei", "😍", "👏", "recomendo", "maravilh"]
    negativos = ["demorou", "péssim", "pessim", "ruim", "horrível", "horrivel",
                 "dinheiro de volta", "cadê", "cade", "não chegou", "nao chegou",
                 "reclam", "decepci", "absurdo", "nunca mais"]
    score = sum(w in t for w in positivos) - sum(w in t for w in negativos)
    if score > 0:
        return "positivo"
    if score < 0:
        return "negativo"
    return "neutro"


def _extrair_texto(prompt: str) -> str:
    """Recupera o texto do cliente que foi injetado no template."""
    m = re.search(r"[Tt]exto:\s*(.+?)(?:\n[Ss]entimento:|\Z)", prompt, re.S)
    if m:
        return m.group(1).strip().strip('"')
    return prompt.rsplit(":", 1)[-1].strip().strip('"')


def _resposta_simulada(texto: str, caps: dict) -> str:
    t = texto.lower()

    # Casos difíceis (negação, frase mista) só saem certos COM exemplos:
    dificeis = {
        "não foi ruim":                ("positivo", "negativo"),  # dupla negação
        "mas a entrega foi péssima":   ("negativo", "positivo"),  # sentimento misto
    }
    for chave, (correta, sem_exemplo) in dificeis.items():
        if chave in t:
            return correta if caps["exemplos"] else sem_exemplo

    base = _heuristica_sentimento(texto)

    # Sem a classe 'neutro' definida, o modelo é forçado a chutar um extremo:
    if base == "neutro" and not caps["neutro"]:
        base = "positivo"
    return base


def _chamar_llm_simulado(prompt: str) -> str:
    """Simulador determinístico, usado quando BACKEND == 'simulador'."""
    caps  = _detectar_capacidades(prompt)
    texto = _extrair_texto(prompt)
    return _resposta_simulada(texto, caps)


def _normalizar_rotulo(saida: str) -> str:
    """Extrai um rótulo válido da resposta (crua) de um modelo real."""
    s = (saida or "").lower()
    for c in ("positivo", "negativo", "neutro"):
        if c in s:
            return c
    return s.strip()


# ------------------- BACKENDS REAIS (opcionais) -------------------
def chamar_llm_gemini(prompt: str) -> str:
    from google import genai                       # pip install google-genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY or None)  # ou GEMINI_API_KEY/GOOGLE_API_KEY do ambiente
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0, max_output_tokens=8),
    )
    return (resp.text or "").strip().lower()


def chamar_llm_anthropic(prompt: str) -> str:
    from anthropic import Anthropic                 # pip install anthropic
    client = Anthropic()                            # usa ANTHROPIC_API_KEY do ambiente
    msg = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=10,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text.strip().lower()


def chamar_llm_azure(prompt: str) -> str:
    from openai import AzureOpenAI                  # pip install openai
    client = AzureOpenAI(api_version="2024-10-21")  # + azure_endpoint / credencial (ou Managed Identity)
    resp = client.chat.completions.create(
        model="gpt-4o-mini", max_tokens=10,         # nome do seu deployment
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content.strip().lower()


_BACKENDS = {"gemini": chamar_llm_gemini, "anthropic": chamar_llm_anthropic, "azure": chamar_llm_azure}


def chamar_llm(prompt: str) -> str:
    """Ponto ÚNICO de chamada ao modelo. Troque BACKEND na 1ª célula para usar um LLM real.
    Todo o resto do notebook (avaliação, A/B, guardrails) usa esta função, sem alteração."""
    t0 = time.perf_counter()
    if BACKEND == "simulador":
        resposta = _chamar_llm_simulado(prompt)
        latencia = LATENCIA_BASE_MS + _estimar_tokens(prompt) * MS_POR_TOKEN   # estimada (reprodutível)
    else:
        resposta = _normalizar_rotulo(_BACKENDS[BACKEND](prompt))
        latencia = (time.perf_counter() - t0) * 1000                          # latência real medida
    chamar_llm.ultima_latencia_ms = round(latencia, 1)
    chamar_llm.ultimos_tokens     = _estimar_tokens(prompt)
    return resposta


print(f"✅ Aurora on-line (backend = '{BACKEND}'). Teste rápido:")
print("   'Adorei!'      ->", chamar_llm("Classifique: Adorei!"))
print("   'Demorou muito'->", chamar_llm("Classifique: Demorou muito"))

✅ Aurora on-line (backend = 'simulador'). Teste rápido:
   'Adorei!'      -> positivo
   'Demorou muito'-> negativo


## 1️⃣ Passo 1 — Definir a métrica

> *"O que significa sucesso? Escolha 1–2 métricas primárias."* — (slide)

Antes de mexer no prompt, a Aurora precisa saber **o que é "bom"**. Sem métrica,
otimização vira achismo: "ah, esse prompt parece melhor". Parece pra quem?

Para classificação de sentimento, a métrica primária natural é a **acurácia** —
a fração de mensagens que a Aurora classifica igual ao gabarito.

- **Primária:** acurácia (quero acertar o rótulo).
- **Secundárias:** custo por mensagem e latência (quero que seja viável em
  escala — milhares de mensagens/dia têm preço).

E um **threshold** combinado com o time: *só vai pra produção quem passar de 85%
de acurácia.* Esse número não é sagrado; é um acordo com os stakeholders.

In [2]:
def acuracia(previsoes, gabaritos) -> float:
    """Métrica primária: fração de acertos."""
    certos = sum(p == g for p, g in zip(previsoes, gabaritos))
    return certos / len(gabaritos)


THRESHOLD_PRODUCAO = 0.85   # acordo com os stakeholders: mínimo para ir a produção
METRICA_PRIMARIA   = "acuracia"

print(f"Métrica primária : {METRICA_PRIMARIA}")
print(f"Threshold mínimo : {THRESHOLD_PRODUCAO:.0%}")
print("Métricas secundárias: custo médio e latência média por mensagem")

Métrica primária : acuracia
Threshold mínimo : 85%
Métricas secundárias: custo médio e latência média por mensagem


## 2️⃣ Passo 2 — Criar o dataset de avaliação

> *"20–50 exemplos com ground truth. Inclua casos limite, ambíguos e
> adversariais."* — (slide)

Aqui está o pulo do gato que quase todo mundo pula: **o dataset é o juiz**. Se o
juiz só tem casos fáceis, qualquer prompt "passa" — e aí a Aurora vai tropeçar
no mundo real.

Por isso o dataset da TechZé mistura de propósito:

- **Casos fáceis** — "Adorei o atendimento!" (positivo óbvio).
- **Casos neutros** — "Recebi o produto hoje" (nem elogio nem reclamação).
- **Casos adversariais** — "Não foi ruim" (dupla negação: parece negativo, é
  positivo).
- **Casos ambíguos/mistos** — "O produto é bom, mas a entrega foi péssima".
- **Frustração implícita** — "Cadê meu pedido???" (sem xingar, mas irritado).

Cada exemplo tem seu **`label`** (o gabarito), idealmente revisado por alguém que
entende do domínio de atendimento.

> Para a aula usamos 10 exemplos enxutos e didáticos. Na vida real, mire nos
> 20–50 do slide (ou mais).

In [3]:
DATASET_VERSAO = "1.0"   # datasets também são versionados!

dataset = [
    {"texto": "Adorei o atendimento!",                          "label": "positivo", "tipo": "fácil"},
    {"texto": "Demorou muito para responder",                   "label": "negativo", "tipo": "fácil"},
    {"texto": "Recebi o produto hoje",                          "label": "neutro",   "tipo": "neutro"},
    {"texto": "Vocês são os melhores 😍",                        "label": "positivo", "tipo": "emoji"},
    {"texto": "Quero meu dinheiro de volta",                    "label": "negativo", "tipo": "fácil"},
    {"texto": "Ok, obrigado",                                   "label": "neutro",   "tipo": "neutro"},
    {"texto": "Não foi ruim",                                   "label": "positivo", "tipo": "adversarial"},
    {"texto": "O produto é bom, mas a entrega foi péssima",     "label": "negativo", "tipo": "ambíguo"},
    {"texto": "Cadê meu pedido???",                             "label": "negativo", "tipo": "frustração"},
    {"texto": "Qual o horário de funcionamento?",              "label": "neutro",   "tipo": "neutro"},
]

df_dataset = pd.DataFrame(dataset)
print(f"Dataset v{DATASET_VERSAO} — {len(dataset)} exemplos")
print("Distribuição de rótulos:", df_dataset["label"].value_counts().to_dict())
df_dataset

Dataset v1.0 — 10 exemplos
Distribuição de rótulos: {'negativo': 4, 'positivo': 3, 'neutro': 3}


,texto,label,tipo
0,Adorei o atendimento!,positivo,fácil
1,Demorou muito para responder,negativo,fácil
2,Recebi o produto hoje,neutro,neutro
3,Vocês são os melhores 😍,positivo,emoji
4,Quero meu dinheiro de volta,negativo,fácil
5,"Ok, obrigado",neutro,neutro
6,Não foi ruim,positivo,adversarial
7,"O produto é bom, mas a entrega foi péssima",negativo,ambíguo
8,Cadê meu pedido???,negativo,frustração
9,Qual o horário de funcionamento?,neutro,neutro


## 3️⃣ Passo 3 — Baseline (v1.0)

> *"Prompt inicial simples, zero-shot. Meça performance no dataset. Este é o
> ponto de referência."* — (slide)

A primeira versão da Aurora é a mais ingênua possível — a que a gente escreve com
pressa no dia 1. É o **`PROMPT_V1`** do slide: binário, sem exemplos, sem sequer
mencionar a classe `neutro`.

Não é para ser bom. É para ser o **ponto de partida** contra o qual tudo será
comparado. Sem baseline, você não sabe se "melhorou".

In [4]:
# --- PROMPT_V1: baseline ingênuo (idêntico ao slide) ---
PROMPT_V1 = "Classifique este texto como positivo ou negativo: {texto}"


def avaliar(prompt_template: str, dataset: list[dict]) -> float:
    """Versão fiel ao slide: só devolve a acurácia."""
    acertos = 0
    for item in dataset:
        resp = chamar_llm(prompt_template.format(texto=item["texto"]))
        if resp.lower().strip() == item["label"]:
            acertos += 1
    return acertos / len(dataset)


baseline = avaliar(PROMPT_V1, dataset)
print(f"📊 Baseline v1.0 — acurácia: {baseline:.0%}")
print(f"   Threshold de produção: {THRESHOLD_PRODUCAO:.0%}  ->",
      "PASSOU ✅" if baseline >= THRESHOLD_PRODUCAO else "REPROVOU ❌ (esperado!)")

📊 Baseline v1.0 — acurácia: 50%
   Threshold de produção: 85%  -> REPROVOU ❌ (esperado!)


**50%.** Um chute entre duas opções acertaria quase isso. 😅

A Aurora v1.0 está reprovada — e tudo bem, esse era o plano. Agora temos um número
concreto para superar. Vamos entender **onde** ela erra para saber **o que**
mudar.

In [5]:
def avaliar_detalhado(prompt_template: str, dataset: list[dict]) -> dict:
    """Além da acurácia, registra erros, custo e latência — as métricas secundárias."""
    acertos, erros = 0, []
    tokens_total, latencia_total = 0, 0.0
    for item in dataset:
        resp = chamar_llm(prompt_template.format(texto=item["texto"]))
        tokens_total   += chamar_llm.ultimos_tokens
        latencia_total += chamar_llm.ultima_latencia_ms
        if resp.lower().strip() == item["label"]:
            acertos += 1
        else:
            erros.append({"texto": item["texto"], "gabarito": item["label"], "aurora": resp})
    n = len(dataset)
    tokens_medio = tokens_total / n
    return {
        "acuracia":     acertos / n,
        "erros":        erros,
        "tokens_medio": round(tokens_medio, 1),
        "custo_1k_msgs": round(tokens_medio * PRECO_POR_1K_TOKENS, 3),  # R$ por 1.000 msgs
        "latencia_ms":  round(latencia_total / n, 1),
    }


res_v1 = avaliar_detalhado(PROMPT_V1, dataset)
print("A v1.0 errou nestes casos:\n")
for e in res_v1["erros"]:
    print(f"  ❌ {e['texto']!r}")
    print(f"       gabarito={e['gabarito']}  |  Aurora disse={e['aurora']}\n")

A v1.0 errou nestes casos:

  ❌ 'Recebi o produto hoje'
       gabarito=neutro  |  Aurora disse=positivo

  ❌ 'Ok, obrigado'
       gabarito=neutro  |  Aurora disse=positivo

  ❌ 'Não foi ruim'
       gabarito=positivo  |  Aurora disse=negativo

  ❌ 'O produto é bom, mas a entrega foi péssima'
       gabarito=negativo  |  Aurora disse=positivo

  ❌ 'Qual o horário de funcionamento?'
       gabarito=neutro  |  Aurora disse=positivo



Olhando os erros, dois padrões saltam:

1. **Ela não sabe dizer "neutro".** Como o prompt só oferece *positivo* ou
   *negativo*, ela é obrigada a chutar um extremo para "Ok, obrigado" e "Recebi o
   produto hoje". → **Hipótese:** *definir a classe `neutro`.*
2. **Ela tropeça em negação e frases mistas** ("Não foi ruim", "...mas a entrega
   foi péssima"). → **Hipótese:** *mostrar exemplos (few-shot).*

Cada padrão vira uma **hipótese testável**. É isso que o próximo passo faz.

## 4️⃣ Passo 4 — Iterar com hipóteses (uma variável por vez)

> *"v1.1: adicionar few-shot. v1.2: reforçar instruções. v1.3: schema control.
> **Uma variável por vez.**"* — (slide)

A regra de ouro: **mude uma coisa de cada vez.** Se você adicionar exemplos *e*
reescrever as instruções *e* trocar o formato tudo junto e a acurácia subir, você
não sabe **o que** funcionou. Ciência de prompt é como experimento controlado.

Nossas versões:

| Versão | Hipótese testada | O que muda |
|--------|------------------|------------|
| **v1.1** | "Falta a classe neutro" | adiciona `neutro` às opções (ainda zero-shot) |
| **v1.2** | "Falta few-shot" | adiciona 3 exemplos (é o `PROMPT_V2` do slide) |
| **v1.3** | "A saída vem bagunçada" | *schema control*: exige resposta de uma palavra |

In [6]:
# v1.1 — hipótese: definir a classe neutro (uma mudança só)
PROMPT_V1_1 = (
    "Classifique o sentimento do texto em: positivo, negativo ou neutro.\n\n"
    "Texto: {texto}\n"
    "Sentimento:"
)

# v1.2 — hipótese: adicionar few-shot  (idêntico ao PROMPT_V2 do slide)
PROMPT_V2 = """Classifique o sentimento do texto em: positivo, negativo ou neutro.

Exemplos:
- "Adorei o atendimento!" -> positivo
- "Demorou muito" -> negativo
- "Recebi o produto" -> neutro

Texto: {texto}
Sentimento:"""

# v1.3 — hipótese: schema control (formato de saída estável para produção)
PROMPT_V1_3 = PROMPT_V2 + "\n\n(Responda apenas com uma palavra, em minúsculas, sem explicações.)"

versoes = {
    "v1.0 (baseline)":     PROMPT_V1,
    "v1.1 (+ neutro)":     PROMPT_V1_1,
    "v1.2 (+ few-shot)":   PROMPT_V2,
    "v1.3 (+ schema)":     PROMPT_V1_3,
}
print("4 versões prontas para o teste A/B:", list(versoes.keys()))

4 versões prontas para o teste A/B: ['v1.0 (baseline)', 'v1.1 (+ neutro)', 'v1.2 (+ few-shot)', 'v1.3 (+ schema)']


## 🆚 Teste A/B — rodando todas as versões no mesmo dataset

É o coração dos slides 1 e 2: **mesmo dataset, mesmo juiz, prompts diferentes.**
Quem acerta mais vence — e agora com número, não com opinião.

Vamos medir cada versão em **acurácia** (primária) e também **custo** e
**latência** (secundárias), porque em produção o prompt mais caro nem sempre
compensa.

In [7]:
resultados = {}
for nome, tpl in versoes.items():
    resultados[nome] = avaliar_detalhado(tpl, dataset)

for nome, r in resultados.items():
    barra = "█" * int(r["acuracia"] * 20)
    print(f"{nome:20s}  {r['acuracia']:>4.0%}  {barra}")

v1.0 (baseline)        50%  ██████████
v1.1 (+ neutro)        80%  ████████████████
v1.2 (+ few-shot)     100%  ████████████████████
v1.3 (+ schema)       100%  ████████████████████


O gráfico de barras já conta a história: **50% → 80% → 100%.**

- **v1.1** deu o maior salto sozinha: só definir a classe `neutro` resolveu 3
  erros de uma vez. Barato e eficaz.
- **v1.2** (few-shot) fechou os casos difíceis de negação e sentimento misto.
- **v1.3** mantém 100% e ainda **estabiliza o formato de saída** — o que importa
  muito quando o próximo passo do sistema depende de receber exatamente uma
  palavra.

Mas atenção: acurácia não é o único eixo. Vamos ver o custo.

## 5️⃣ Passo 5 — Comparar e documentar

> *"Tabela com versões × métricas. Incluir custo e latência. Promover versão
> vencedora a produção."* — (slide)

Aqui a decisão deixa de ser técnica e vira **produto**: qual versão vale a pena
levar a produção, considerando acurácia **e** o custo de rodar isso milhares de
vezes por dia?

In [8]:
linhas = []
for nome, r in resultados.items():
    linhas.append({
        "Versão":            nome,
        "Acurácia":          f"{r['acuracia']:.0%}",
        "Tokens/msg":        r["tokens_medio"],
        "Custo /1k msgs (R$)": r["custo_1k_msgs"],
        "Latência (ms)":     r["latencia_ms"],
        "Passa threshold?":  "✅" if r["acuracia"] >= THRESHOLD_PRODUCAO else "❌",
    })

tabela = pd.DataFrame(linhas)
tabela

,Versão,Acurácia,Tokens/msg,Custo /1k msgs (R$),Latência (ms),Passa threshold?
0,v1.0 (baseline),50%,18.0,0.540,65.2,❌
1,v1.1 (+ neutro),80%,27.6,0.828,78.6,❌
2,v1.2 (+ few-shot),100%,55.0,1.650,117.0,✅
3,v1.3 (+ schema),100%,72.0,2.160,140.8,✅


In [9]:
# Regra de promoção: entre as versões que passam o threshold, escolher a de
# maior acurácia; em empate, preferir a de saída mais estável (schema control);
# só então desempatar por menor custo.
candidatas = {n: r for n, r in resultados.items() if r["acuracia"] >= THRESHOLD_PRODUCAO}

vencedora = max(
    candidatas.items(),
    key=lambda kv: (kv[1]["acuracia"], "schema" in kv[0], -kv[1]["custo_1k_msgs"]),
)[0]

PROMPT_PRODUCAO = versoes[vencedora]

print(f"🏆 Versão promovida a produção: {vencedora}")
print(f"   Acurácia   : {resultados[vencedora]['acuracia']:.0%}")
print(f"   Custo/1k   : R$ {resultados[vencedora]['custo_1k_msgs']}")
print(f"   Latência   : {resultados[vencedora]['latencia_ms']} ms")
print()
print("Decisão documentada: v1.2 e v1.3 empatam em acurácia (100%), mas a v1.3")
print("controla o formato de saída — mais segura para o sistema a jusante. Vence a v1.3.")

🏆 Versão promovida a produção: v1.3 (+ schema)
   Acurácia   : 100%
   Custo/1k   : R$ 2.16
   Latência   : 140.8 ms

Decisão documentada: v1.2 e v1.3 empatam em acurácia (100%), mas a v1.3
controla o formato de saída — mais segura para o sistema a jusante. Vence a v1.3.


### 📝 Registro da decisão (o "documentar" do slide)

> **ADR — Escolha do prompt de classificação de sentimento (Aurora)**
> - **Contexto:** triagem de mensagens de suporte por sentimento.
> - **Baseline:** v1.0 (binário, zero-shot) → 50%. Reprovado.
> - **Iterações:** v1.1 (+neutro, 80%), v1.2 (+few-shot, 100%), v1.3 (+schema, 100%).
> - **Decisão:** promover **v1.3**. Acurácia máxima **e** formato de saída estável.
> - **Trade-off aceito:** ~4× mais tokens que o baseline; justificado pela queda
>   de erros de triagem e pela robustez do parsing.
> - **Próxima revisão:** reavaliar quando o dataset crescer para 50 exemplos ou
>   ao trocar de modelo.

Esse pequeno registro é ouro: daqui a três meses, ninguém vai lembrar *por que* a
v1.3 venceu — mas o documento vai.

## ✅ Checklist para Produção Responsável

> *"Antes de levar qualquer LLM à produção, este checklist deve estar 100%
> coberto."* — (slide)

A Aurora acertou 100% no teste. **Isso não significa que ela está pronta.** Acurácia
alta num dataset pequeno é só o começo. Antes de deixá-la responder clientes de
verdade, seis frentes precisam estar de pé. Vamos **implementar as que dá para
implementar em código** e deixar claras as que são processo/organização.

| # | Item | Natureza |
|---|------|----------|
| 1 | Métricas definidas | ✔️ feito nos passos 1 e 5 |
| 2 | Dataset de avaliação | ✔️ feito no passo 2 (curado, versionado, edge cases) |
| 3 | Guardrails ativos | 🧑‍💻 vamos codar |
| 4 | Observabilidade | 🧑‍💻 vamos codar |
| 5 | Conformidade LGPD | 🧑‍💻 (parte técnica) + processo |
| 6 | Plano de rollback | 🧑‍💻 vamos codar |

### 🛡️ Guardrails + LGPD — protegendo entrada e saída

Dois perigos concretos numa central de atendimento:

- **PII na entrada:** clientes mandam CPF, e-mail, telefone no meio da mensagem.
  A LGPD exige **anonimizar** antes de logar ou enviar a terceiros.
- **Saída fora do contrato:** se o modelo devolver `"acho que é positivo :)"` em
  vez de `positivo`, o sistema a jusante quebra. O guardrail de saída **valida e
  normaliza** o rótulo.

In [10]:
# ---------- LGPD: anonimização de PII na entrada ----------
def mascarar_pii(texto: str) -> str:
    texto = re.sub(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b", "[CPF]", texto)
    texto = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", texto)
    texto = re.sub(r"\b(?:\+?55\s?)?\(?\d{2}\)?\s?9?\d{4}[-\s]?\d{4}\b", "[TELEFONE]", texto)
    return texto


# ---------- Guardrail de ENTRADA ----------
def guardrail_entrada(texto: str) -> tuple[bool, str]:
    if not texto or not texto.strip():
        return False, "mensagem vazia"
    if len(texto) > 2000:
        return False, "mensagem longa demais (possível abuso)"
    return True, "ok"


# ---------- Schema control / parsing robusto da SAÍDA ----------
def parsear_rotulo(saida: str):
    """Extrai um rótulo válido mesmo de uma resposta bagunçada do modelo."""
    s = saida.lower()
    for c in ("positivo", "negativo", "neutro"):
        if c in s:
            return c
    return None


# ---------- Guardrail de SAÍDA ----------
def guardrail_saida(saida: str) -> tuple[str, bool]:
    rotulo = parsear_rotulo(saida)
    if rotulo in CLASSES_VALIDAS:
        return rotulo, True
    return "neutro", False   # fallback seguro + sinaliza para revisão


# --- Demonstração ---
exemplo = "Meu CPF é 123.456.789-00 e email joao@teste.com, quero cancelar! Zap (11) 98888-7777"
print("Original :", exemplo)
print("Anonimizado:", mascarar_pii(exemplo))
print()
print("Saída bagunçada 'Acho que é POSITIVO :)' ->",
      guardrail_saida("Acho que é POSITIVO :)"))
print("Saída inválida  'sei lá'                 ->",
      guardrail_saida("sei lá"))

Original : Meu CPF é 123.456.789-00 e email joao@teste.com, quero cancelar! Zap (11) 98888-7777
Anonimizado: Meu CPF é [CPF] e email [EMAIL], quero cancelar! Zap ([TELEFONE]

Saída bagunçada 'Acho que é POSITIVO :)' -> ('positivo', True)
Saída inválida  'sei lá'                 -> ('neutro', False)


### 🔭 Observabilidade — enxergar o que a Aurora faz em produção

Você não pode melhorar o que não mede em produção. Cada chamada precisa deixar um
**rastro** (trace): qual versão do prompt, o que entrou (já anonimizado!), o que
saiu, quanto custou, quanto demorou.

Aqui fazemos um logger mínimo em memória. Em produção, isso vira **Langfuse** ou
**LangSmith**, alimentando dashboards de latência, custo e taxa de erro.

In [11]:
TRACES = []   # em produção: Langfuse / LangSmith

def observar(evento: dict):
    evento["ts"] = time.strftime("%H:%M:%S")
    TRACES.append(evento)


# ---------- Rollback: interruptor mestre ----------
IA_ATIVA = True   # feature flag: vira False e a IA é desligada instantaneamente

def transferir_para_humano(texto: str) -> dict:
    return {"canal": "humano",
            "rotulo": None,
            "resposta": "Recebemos sua mensagem! Um atendente vai te responder já já 💬"}


# ---------- Pipeline completo de atendimento ----------
def atender(mensagem: str, prompt_template: str = None, versao: str = "prod") -> dict:
    prompt_template = prompt_template or PROMPT_PRODUCAO

    # 6) Rollback tem prioridade máxima
    if not IA_ATIVA:
        r = transferir_para_humano(mensagem)
        observar({"versao": versao, "rollback": True, "canal": "humano"})
        return r

    # 3) Guardrail de entrada
    ok, motivo = guardrail_entrada(mensagem)
    if not ok:
        observar({"versao": versao, "bloqueado_entrada": motivo})
        return {"canal": "humano", "rotulo": None, "resposta": f"(encaminhado: {motivo})"}

    # 5) LGPD: anonimiza antes de qualquer coisa
    msg_segura = mascarar_pii(mensagem)

    # chamada ao modelo (via prompt de produção vencedor)
    t0 = time.perf_counter()
    bruto = chamar_llm(prompt_template.format(texto=msg_segura))
    dt_ms = round((time.perf_counter() - t0) * 1000 + chamar_llm.ultima_latencia_ms, 1)

    # 3) Guardrail de saída + schema control
    rotulo, valido = guardrail_saida(bruto)

    # 4) Observabilidade: registra o trace
    observar({
        "versao": versao,
        "entrada": msg_segura,          # já anonimizada
        "rotulo": rotulo,
        "saida_valida": valido,
        "tokens": chamar_llm.ultimos_tokens,
        "latencia_ms": dt_ms,
    })

    prioridade = "🔴 ALTA" if rotulo == "negativo" else "🟢 normal"
    return {"canal": "ia", "rotulo": rotulo, "prioridade": prioridade, "valido": valido}


# --- Rodando algumas mensagens pela Aurora "de produção" ---
mensagens = [
    "Não foi ruim, gostei bastante!",
    "Meu CPF 111.222.333-44 e não recebi o pedido, cadê???",
    "Ok, obrigado pela ajuda",
]
for m in mensagens:
    print(f"{m[:45]:45s} -> {atender(m)}")

Não foi ruim, gostei bastante!                -> {'canal': 'ia', 'rotulo': 'positivo', 'prioridade': '🟢 normal', 'valido': True}
Meu CPF 111.222.333-44 e não recebi o pedido, -> {'canal': 'ia', 'rotulo': 'negativo', 'prioridade': '🔴 ALTA', 'valido': True}
Ok, obrigado pela ajuda                       -> {'canal': 'ia', 'rotulo': 'neutro', 'prioridade': '🟢 normal', 'valido': True}


Repare que a segunda mensagem foi **anonimizada** antes de ir para o log, e a
mensagem negativa recebeu **prioridade ALTA** — exatamente a missão original da
Aurora: fazer reclamações furarem a fila. 🎯

Vamos ver os traces registrados e simular um **rollback**:

In [12]:
print("📋 Traces observados:")
for t in TRACES:
    print("  ", t)

print("\n--- Simulando incêndio em produção: desligando a IA ---")
IA_ATIVA = False
print(atender("Adorei tudo!"))
IA_ATIVA = True   # religando após o susto
print("IA religada:", atender("Adorei tudo!"))

📋 Traces observados:
   {'versao': 'prod', 'entrada': 'Não foi ruim, gostei bastante!', 'rotulo': 'positivo', 'saida_valida': True, 'tokens': 74, 'latencia_ms': 143.6, 'ts': '23:35:58'}
   {'versao': 'prod', 'entrada': 'Meu CPF [CPF] e não recebi o pedido, cadê???', 'rotulo': 'negativo', 'saida_valida': True, 'tokens': 77, 'latencia_ms': 147.8, 'ts': '23:35:58'}
   {'versao': 'prod', 'entrada': 'Ok, obrigado pela ajuda', 'rotulo': 'neutro', 'saida_valida': True, 'tokens': 72, 'latencia_ms': 140.8, 'ts': '23:35:58'}

--- Simulando incêndio em produção: desligando a IA ---
{'canal': 'humano', 'rotulo': None, 'resposta': 'Recebemos sua mensagem! Um atendente vai te responder já já 💬'}
IA religada: {'canal': 'ia', 'rotulo': 'positivo', 'prioridade': '🟢 normal', 'valido': True}


### 🧾 Verificação automática do checklist

Fechando o ciclo: um teste que confere, programaticamente, se cada item do
checklist está coberto. É o tipo de *smoke test* que pode virar um gate no CI/CD
antes de qualquer deploy.

In [13]:
def verificar_checklist() -> pd.DataFrame:
    checks = []

    # 1) Métricas definidas
    ok = "METRICA_PRIMARIA" in globals() and 0 < THRESHOLD_PRODUCAO <= 1
    checks.append(("1. Métricas definidas", ok, "primária + threshold acordado"))

    # 2) Dataset de avaliação
    tem_edge = any(i["tipo"] in ("adversarial", "ambíguo") for i in dataset)
    ok = len(dataset) >= 10 and tem_edge and bool(DATASET_VERSAO)
    checks.append(("2. Dataset de avaliação", ok, f"v{DATASET_VERSAO}, {len(dataset)} ex., com edge cases"))

    # 3) Guardrails ativos
    try:
        e_ok = guardrail_entrada("teste")[0]
        s_ok = guardrail_saida("positivo")[0] == "positivo"
        ok = e_ok and s_ok
    except Exception:
        ok = False
    checks.append(("3. Guardrails ativos", ok, "entrada + saída validadas"))

    # 4) Observabilidade
    ok = len(TRACES) > 0 and all("latencia_ms" in t or "rollback" in t or "bloqueado_entrada" in t for t in TRACES)
    checks.append(("4. Observabilidade", ok, f"{len(TRACES)} traces (latência/custo/erros)"))

    # 5) Conformidade LGPD (parte técnica)
    ok = "[CPF]" in mascarar_pii("meu cpf 123.456.789-00")
    checks.append(("5. LGPD — PII anonimizada", ok, "técnico ok; falta aprovação do DPO + retention policy"))

    # 6) Plano de rollback
    ok = "IA_ATIVA" in globals() and callable(transferir_para_humano)
    checks.append(("6. Plano de rollback", ok, "feature flag + fallback humano"))

    df = pd.DataFrame(
        [{"Item": n, "Status": "✅" if ok else "❌", "Observação": obs} for n, ok, obs in checks]
    )
    return df


relatorio = verificar_checklist()
cobertura = (relatorio["Status"] == "✅").mean()
print(f"Cobertura do checklist: {cobertura:.0%}\n")
relatorio

Cobertura do checklist: 100%



,Item,Status,Observação
0,1. Métricas definidas,✅,primária + threshold acordado
1,2. Dataset de avaliação,✅,"v1.0, 10 ex., com edge cases"
2,3. Guardrails ativos,✅,entrada + saída validadas
3,4. Observabilidade,✅,5 traces (latência/custo/erros)
4,5. LGPD — PII anonimizada,✅,técnico ok; falta aprovação do DPO + retention...
5,6. Plano de rollback,✅,feature flag + fallback humano


> ⚠️ **Honestidade intelectual:** os itens marcados ✅ são a parte **técnica**.
> "LGPD" de verdade também exige **aprovação do DPO**, definição de *data
> residency* e *retention policy* — decisões organizacionais que nenhum notebook
> resolve sozinho. O código cobre o *como*; o processo cobre o *quem aprova*.

## 🌅 Epílogo — a jornada da Aurora

Começamos com uma assistente ingênua que acertava **metade** das mensagens e teria
mandado "Não foi ruim" para a fila de reclamações. Terminamos com uma Aurora que:

- foi **medida** contra uma métrica e um threshold combinados com o time;
- foi **testada** num dataset com casos difíceis de propósito;
- **evoluiu por hipóteses**, uma variável por vez — 50% → 80% → 100%;
- teve sua versão vencedora **escolhida com dados** (acurácia *e* custo) e
  **documentada**;
- e só então recebeu **guardrails, anonimização LGPD, observabilidade e um botão
  de rollback** antes de encostar num cliente real.

**A moral:** um prompt bom não nasce de inspiração — nasce de **método**. E colocar
IA em produção com responsabilidade é menos sobre o modelo e mais sobre tudo o que
cerca o modelo.

### 🔜 Para levar adiante
- Crescer o dataset para 30–50 exemplos e adicionar métricas por classe
  (precision/recall), não só acurácia global.
- Trocar o simulador pelo **Google Gemini** (basta `BACKEND = "gemini"` — ver o
  final do notebook) e re-rodar o teste A/B com números reais.
- Ligar as traces a um **Langfuse/LangSmith** de verdade.
- Transformar `verificar_checklist()` num gate de CI/CD antes do deploy.

## 🔌 Usando o Google AI (Gemini) — ou outro modelo — de verdade

Tudo acima rodou com o **simulador**, para a aula ser reprodutível. Para usar um
modelo real **você não muda nada no meio do notebook**: só troca o `BACKEND` lá na
**primeira célula de código**. Toda a lógica (avaliação, A/B, guardrails, checklist)
chama o mesmo `chamar_llm()` — essa é a beleza de separar o *prompt* da *chamada*.

**Para ligar o Google AI / Gemini:**

1. `pip install google-genai`
2. Pegue a chave em <https://aistudio.google.com/apikey> e configure
   `export GEMINI_API_KEY="sua-chave"` (ou cole em `GEMINI_API_KEY` na 1ª célula).
3. Na 1ª célula de código, mude para **`BACKEND = "gemini"`**.
4. **Kernel → Restart & Run All.** O mesmo teste A/B agora roda no Gemini e os
   números da tabela passam a ser **reais** (acurácia do modelo, latência medida).

> As funções `chamar_llm_anthropic` (Claude) e `chamar_llm_azure` (Azure OpenAI,
> compatível com Managed Identity) já estão prontas — é só apontar o `BACKEND`.

A célula abaixo é um *smoke test*: com o simulador, imprime o passo a passo; com um
backend real configurado, classifica algumas mensagens **ao vivo**.

In [14]:
# Smoke test do backend (roda com qualquer configuração)
if BACKEND == "simulador":
    print("Backend atual: SIMULADOR (padrão, sem chave).\n")
    print("Para classificar com o Google AI / Gemini de verdade:")
    print("  1) pip install google-genai")
    print("  2) export GEMINI_API_KEY='sua-chave'   ->  https://aistudio.google.com/apikey")
    print("  3) na 1ª célula de código:  BACKEND = 'gemini'")
    print("  4) Kernel > Restart & Run All")
else:
    alvo = GEMINI_MODEL if BACKEND == "gemini" else BACKEND
    print(f"Backend atual: {BACKEND.upper()} ({alvo}) — classificando ao vivo:\n")
    for m in ["Adorei o atendimento!", "Demorou demais para chegar", "Recebi o produto hoje"]:
        print(f"  {m!r:35s} -> {chamar_llm(PROMPT_PRODUCAO.format(texto=m))}")

Backend atual: SIMULADOR (padrão, sem chave).

Para classificar com o Google AI / Gemini de verdade:
  1) pip install google-genai
  2) export GEMINI_API_KEY='sua-chave'   ->  https://aistudio.google.com/apikey
  3) na 1ª célula de código:  BACKEND = 'gemini'
  4) Kernel > Restart & Run All
